# Fase 3.8 - AIGI pareado, execucao manual no Colab

O piloto AIGI foi concluido. Esta versao adiciona pares reais/sinteticos por UID, seis geradores e o mesmo JPEG 256 para ambas as classes. O objetivo e reduzir atalhos de formato e tamanho no novo conjunto; isso nao comprova ganho externo. Mantenha os resultados antigos. Execute 1-5, depois preparo (7), copia local (8) e smoke (9). A celula 10 inicia ou retoma o treino apenas quando executada manualmente. Depois leia 11, copie Mirage (12), congele (13), avalie (14) e leia (15). Nao use Run All.


In [ ]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

In [ ]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_8', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

In [ ]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

In [ ]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase38-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_8_aigi_piloto.py', 'experimentos/fase3_8_aigi_dados.py', 'experimentos/fase3_8_aigi_finetune.py', 'experimentos/avaliar_fase3_8_mirage.py', 'experimentos/fase3_5_dados.py', 'experimentos/fase3_7_dados.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.8 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

In [ ]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_8_aigi_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_8_aigi_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

In [ ]:
# 6. PILOTO JA EXECUTADO: a saida foi analisada antes deste plano.
print('Piloto AIGI: commit fixado; 16 fotos reais testadas, 15 acessiveis, somente 2 com lado curto >=512. O preparo usa lado curto >=256 e valida cada par.')


In [ ]:
# 7. PREPARO MANUAL: plano original + tres shards adicionais, sem perder os pares aceitos.
# Usa a credencial HF cifrada somente para o download, quando disponivel.
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
hf_env = ENV.copy()
hf_path = ROOT / 'credenciais/huggingface.enc'
if hf_path.is_file():
    credential = json.loads(hf_path.read_text(encoding='utf-8'))
    if credential.get('version') != 1 or not {'salt', 'iterations', 'ciphertext'} <= credential.keys():
        raise ValueError('Formato da credencial Hugging Face nao reconhecido')
    if int(credential['iterations']) < 600000:
        raise ValueError('Credencial Hugging Face com KDF insuficiente')
    password = getpass.getpass('Senha de desbloqueio do Hugging Face: ')
    key = base64.urlsafe_b64encode(PBKDF2HMAC(
        algorithm=hashes.SHA256(), length=32,
        salt=base64.b64decode(credential['salt']),
        iterations=int(credential['iterations'])).derive(password.encode()))
    try:
        secret = Fernet(key).decrypt(credential['ciphertext'].encode()).decode()
    except InvalidToken:
        raise ValueError('Senha incorreta ou credencial Hugging Face corrompida') from None
    finally:
        del password, key
    hf_env['HF_TOKEN'] = secret
    del secret
    print('Credencial Hugging Face desbloqueada para esta preparacao.', flush=True)
else:
    print('Credencial Hugging Face cifrada ausente; AIGI publico sera acessado sem autenticacao.', flush=True)
try:
    run_live('preparacao 3.8', [sys.executable, '-u', str(DATA), 'prepare',
                                '--root', str(ROOT)], child_env=hf_env)
finally:
    hf_env.pop('HF_TOKEN', None)


In [ ]:
# 8. COPIA LOCAL: execute a cada nova sessao antes do smoke/treino.
OLD_DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
PRIOR_DATA = CODE_ROOT / 'experimentos/fase3_7_dados.py'
run_live('copia local 3.5', [sys.executable, '-u', str(OLD_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local 3.7', [sys.executable, '-u', str(PRIOR_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local AIGI 3.8', [sys.executable, '-u', str(DATA), 'stage', '--root', str(ROOT)])


In [ ]:
# 9. SMOKE MANUAL: verifica particoes, hashes, GPU, checkpoint, forward e escrita; nao treina.
run_live('smoke 3.8', [sys.executable, '-u', str(TRAIN), 'smoke'])


In [ ]:
# 10. TREINO MANUAL: so execute apos Smoke OK; auto inicia ou retoma de last.pt.
run_live('ajuste fino 3.8', [sys.executable, '-u', str(TRAIN), 'auto'])


In [ ]:
# 11. LEITURA INTERNA: selecao escolhe epoca; calibracao define T e limiar.
run_dir = ROOT / 'modelos/fase3_8_aigi_pares256_v1'
for name in ('experiment_config.json', 'split_manifest.json', 'history.csv', 'last.pt',
             'best.pt', 'phase3_final.pt', 'internal_metrics.json', 'training_complete.json'):
    print(name, 'presente' if (run_dir / name).is_file() else 'AUSENTE')
metrics = run_dir / 'internal_metrics.json'
print(json.dumps(json.loads(metrics.read_text(encoding='utf-8')), indent=2)
      if metrics.is_file() else 'Treino interno ainda nao concluido.')


## Avaliacao externa comparativa

As celulas 12-15 copiam as mesmas 2.000 imagens Mirage ja congeladas, registram checkpoint, temperatura e limiar antes da inferencia e preservam o resultado da 3.8 em pasta propria. Mirage ja foi observado e nao deve orientar ajustes de treino ou calibracao.


In [ ]:
# 12. MIRAGE: copia local verificada, sem novo download da fonte.
MIRAGE_DATA = CODE_ROOT / 'experimentos/benchmark_mirage_2000.py'
run_live('copia local Mirage', [sys.executable, '-u', str(MIRAGE_DATA), 'stage', '--root', str(ROOT)])


In [ ]:
# 13. CONGELAMENTO: checkpoint, temperatura, limiar, codigo e manifesto.
EVAL = CODE_ROOT / 'experimentos/avaliar_fase3_8_mirage.py'
run_live('congelamento 3.8', [sys.executable, '-u', str(EVAL), 'freeze', '--root', str(ROOT)])


In [ ]:
# 14. AVALIACAO EXTERNA COMPARATIVA apos o congelamento.
run_live('avaliacao Mirage 3.8', [sys.executable, '-u', str(EVAL), 'evaluate', '--root', str(ROOT)])


In [ ]:
# 15. LEITURA do resultado preservado; sem recalibracao.
result = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_8_aigi_pares256_v1/metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2)
      if result.is_file() else 'Avaliacao externa ainda nao executada.')
